In [14]:
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV
)

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import (
    mean_squared_error,
    r2_score
)

In [15]:
# =====================================================
# STEP 1: LOAD DATASET
# =====================================================

data = fetch_california_housing(as_frame=True)

df = pd.concat(
    [data.data,
     data.target.rename("HousePrice")],
    axis=1
)

print("Dataset Shape:", df.shape)


Dataset Shape: (20640, 9)


In [16]:
# =====================================================
# STEP 2: FEATURES & TARGET
# =====================================================
X = df.drop("HousePrice", axis=1)
y = df["HousePrice"]

In [17]:
# =====================================================
# STEP 3: FEATURE SCALING
# =====================================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [18]:
# =====================================================
# STEP 4: TRAIN TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)


In [19]:
# =====================================================
# STEP 5: LINEAR REGRESSION
# =====================================================

lr = LinearRegression()

lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2 = r2_score(y_test, lr_pred)

print("\nLinear Regression")
print("RMSE:", lr_rmse)
print("R2:", lr_r2)


Linear Regression
RMSE: 0.7455813830127763
R2: 0.575787706032451


In [20]:
# =====================================================
# STEP 6: RIDGE REGRESSION
# =====================================================

ridge = Ridge(alpha=1.0)

ridge.fit(X_train, y_train)

ridge_pred = ridge.predict(X_test)

ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
ridge_r2 = r2_score(y_test, ridge_pred)

print("\nRidge Regression")
print("RMSE:", ridge_rmse)
print("R2:", ridge_r2)


Ridge Regression
RMSE: 0.7455542909384605
R2: 0.5758185345441327


In [21]:
# =====================================================
# STEP 7: OVERFITTING DETECTION
# =====================================================

tree = DecisionTreeRegressor(random_state=42)

tree.fit(X_train, y_train)

train_pred = tree.predict(X_train)
test_pred = tree.predict(X_test)

train_rmse = np.sqrt(
    mean_squared_error(y_train, train_pred)
)

test_rmse = np.sqrt(
    mean_squared_error(y_test, test_pred)
)

print("\nDecision Tree Overfitting Check")
print("Training RMSE:", train_rmse)
print("Testing RMSE :", test_rmse)


Decision Tree Overfitting Check
Training RMSE: 3.218325866275131e-16
Testing RMSE : 0.7030445773467542


In [22]:
# =====================================================
# STEP 8: CROSS VALIDATION
# =====================================================

cv_scores = cross_val_score(
    tree,
    X_scaled,
    y,
    scoring="neg_root_mean_squared_error",
    cv=5
)

cv_rmse = -cv_scores.mean()

print("\nCross Validation RMSE:", cv_rmse)


Cross Validation RMSE: 0.8957031908951016


In [23]:
# =====================================================
# STEP 9: HYPERPARAMETER TUNING
# =====================================================

param_grid = {
    "max_depth": [3, 5, 7, 10],
    "min_samples_split": [2, 5, 10]
}

grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("\nBest Parameters")
print(grid.best_params_)


Best Parameters
{'max_depth': 10, 'min_samples_split': 10}


In [24]:
# =====================================================
# STEP 10: EVALUATE BEST MODEL
# =====================================================

best_tree = grid.best_estimator_

best_pred = best_tree.predict(X_test)

tree_rmse = np.sqrt(
    mean_squared_error(y_test, best_pred)
)

tree_r2 = r2_score(y_test, best_pred)

print("\nTuned Decision Tree")
print("RMSE:", tree_rmse)
print("R2:", tree_r2)




Tuned Decision Tree
RMSE: 0.6454300828015771
R2: 0.6820992539714815


In [25]:
# =====================================================
# STEP 11: MODEL COMPARISON TABLE
# =====================================================

results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge Regression",
        "Tuned Decision Tree"
    ],
    "RMSE": [
        lr_rmse,
        ridge_rmse,
        tree_rmse
    ],
    "R2 Score": [
        lr_r2,
        ridge_r2,
        tree_r2
    ]
})

print("\nModel Comparison")
print(results)


Model Comparison
                 Model      RMSE  R2 Score
0    Linear Regression  0.745581  0.575788
1     Ridge Regression  0.745554  0.575819
2  Tuned Decision Tree  0.645430  0.682099


In [26]:
# =====================================================
# STEP 12: BEST MODEL
# =====================================================

best_model = results.loc[
    results["RMSE"].idxmin()
]

print("\nBest Model:")
print(best_model)


Best Model:
Model       Tuned Decision Tree
RMSE                    0.64543
R2 Score               0.682099
Name: 2, dtype: object
